# Jev / Tev analysis starter
Load saved results; do the analysis yourself below. [Definitions and test guidance](README.md).
No gold labels are provided. Metric calculations and model tests are intentionally left blank.

In [ ]:
import sys
import pandas as pd
from IPython.display import display
from jev_bench.run_evaluations.comparison import load_source
from jev_bench.storage.paths import PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT))
from analysis.stats_helpers import welch, levene, mean_power

RESULTS = PROJECT_ROOT / "results" / "experiments"
assert sys.version_info[:2] == (3, 12), "Select the repository's Python 3.12 .venv kernel"

In [ ]:
old_sources = ["tev_30", "gemma_30", "jev_api_30", "baseline"]
new_sources = [f"refactor_validation_20261003_{model}" for model in [
    "tev1_0.8b", "tev1_4b", "gemma4_e4b", "jev-1.13.0",
    "mistral-small-latest", "mistral-large-latest",
]]


def load_suite(suite):
    frames = []
    for cohort, sources in [("old", old_sources), ("new", new_sources)]:
        for source in sources:
            directory = RESULTS / source / suite
            if not (directory / "raw.csv").exists():
                print(f"Missing suite: {source}/{suite} (no measurements loaded)")
                continue  # The new Mistral hard-case runs have no CSVs.
            frame = load_source(directory, suite)["frame"]
            if source == "baseline":
                frame = frame[frame.model.isin(["mistral-small-latest", "mistral-large-latest"])]
            frames.append(frame.assign(source_dataset=source, cohort=cohort))
    return pd.concat(frames, ignore_index=True)


df_benchmark10 = load_suite("benchmark")
df_hardcase = load_suite("hard_case")

Combined in memory only. Original repetition IDs are kept; `source_dataset` distinguishes overlapping IDs.
Rejected rows stay visible. Probability/classification analysis uses accepted rows (`validation_success`);
this flag means valid execution/output, **not** correct classification. Resource totals including retries
need the audited reader's `history` frame as well. Mistral's counts are incomplete.

In [ ]:
for name, frame, keys in [
    ("benchmark10", df_benchmark10, ["model", "case_id"]),
    ("hardcase", df_hardcase, ["model"]),
]:
    print(name)
    display(frame.groupby(keys).agg(observed=("validation_success", "size"),
                                  accepted=("validation_success", "sum")))
    display(frame.groupby(["cohort", *keys]).agg(observed=("validation_success", "size"),
                                               accepted=("validation_success", "sum")))

## Mistral: retain data, exploratory variance analysis only
Mistral rows remain in both DataFrames for transparency. **Exclude Mistral from the primary stability comparison**: accepted repetitions per question are too sparse, and the hard case has only one accepted call per model.

Never include rejected cached/cache-unverified rows in probability or variance tests. Accepted Mistral rows passed the existing cache checks; any analysis of those rows is **exploratory**, using actual per-question counts. Do not treat pooled calls across different questions as repetitions of one question.

The diagnostic below reports cache and other failures separately; it performs no model comparisons.


In [ ]:
for name, frame in [("benchmark10", df_benchmark10), ("hardcase", df_hardcase)]:
    mistral = frame[frame.model.str.startswith("mistral-")]
    print(f"Mistral: {name}")
    display(mistral.assign(failure_kind=mistral.failure_kind.replace("", "none"))
            .groupby(["model", "validation_success", "failure_kind"], dropna=False)
            .size().rename("calls").to_frame())


## Metrics / comparisons — your work
- **Accuracy:** your labels; binary threshold `p >= 0.50`. Define route/freshness references separately.
- **Repeatability:** decision agreement and probability spread on a fixed input/field; correct repetitions measure accuracy.
- **Absolute probability error:** `abs(p - y)`. **Decisiveness:** distance from 0.5, not proven confidence.
- **Efficiency:** latency and input/output tokens alongside accuracy; count failed/retried work too.

Welch compares independent continuous means; median-centered Levene compares spread for one input/field.
Use paired question-level comparisons for benchmark accuracy. Check cohorts and distributions before pooling;
use actual counts, alpha 0.05, Holm adjustment, effect sizes and confidence intervals. Non-significance is not equality.
Forty calls per question are still only **ten distinct benchmark questions** and **one hard case**.
Local cold latency and hosted wall time have different boundaries; Jev caching is unverified.

In [ ]:
# Your labels and metric calculations here.

In [ ]:
# Your selected comparisons here: welch(a, b), levene(a, b).

## Synthetic mean-effect power sensitivity
Illustrative assumptions only; no saved model outcomes are tested.
Independent: 40 calls **per group** for a fixed input, effect standardized by a common SD (equal-variance approximation, not exact Welch power).
Paired: ten independent question pairs, effect standardized by the SD of paired differences.
The effect scales differ. Power is detection probability under an assumed effect, not observed significance;
this table does not cover accuracy, variance tests, or incomplete/unequal Mistral samples.

In [ ]:
pd.DataFrame([
    {"assumed_effect": effect,
     "power_40_calls_per_group": mean_power(effect, n=40),
     "power_10_question_pairs": mean_power(effect, n=10, paired=True)}
    for effect in [0.2, 0.5, 0.8]
])